# EB-NeRD — iteration 1  (CPU session)

Attach your `ebnerd_large_dataset` and run top to bottom. No GPU: Ekstra Bladet supplies
pre-computed article embeddings, which this notebook downloads from their public S3 bucket.

**This is not a copy of the MIND notebook.** EB-NeRD differs in ways that would break a
straight port, and several of them are silent failures rather than errors:

| | MIND | EB-NeRD |
|---|---|---|
| Submission filename | `prediction.txt` | **`predictions.txt`** (plural) |
| Daily submissions | 10 | **5** |
| Format | parquet? no — TSV | Parquet |
| History | inline on every impression row | **separate file**, one row per user |
| Labels | glued to the candidate (`N123-1`) | **separate list**, must be intersected |
| Publication time | absent (we reconstructed a proxy) | **supplied and exact** |
| Popularity stats | none | **`total_pageviews` / `total_inviews` supplied** |
| Article embeddings | we had to encode them | **supplied as artifacts** |

**What we already measured on the demo bundle**, which drives the feature choices below:

- `total_pageviews` alone scores **AUC 0.5969** — the strongest single feature available.
- Article age is **strongly non-monotonic**: click rate peaks at 2–4 h and is *lower* under
  1 h, with an **8.1x** spread between best and worst bucket. Yet a linear `-age` term
  scores **0.4950 — below random.** Exactly the trap that cost us a feature on MIND, but
  with a bigger prize here, so age enters as a bucketed lookup.
- `sentiment_score` scores 0.5054, i.e. nothing. Included only as an ablation row.

## 1. Setup, data discovery, and the embedding artifact

In [23]:
import gc, glob, json, re, time, warnings, zipfile, urllib.request
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import polars as pl
import scipy.sparse as sp

import os, shutil, psutil

# A2 NRMS dependencies
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
WORK = Path("/kaggle/working")
_PROC = psutil.Process(os.getpid())


def mem(tag=""):
    """Resident memory and free disk. Kaggle gives ~30 GB RAM on CPU but only ~13 GB on a
    P100 GPU session; /kaggle/working is separately capped, and the first full run died of
    ENOSPC rather than OOM, so both numbers are worth watching."""
    print(f"   [mem] {_PROC.memory_info().rss/1e9:5.2f} GB | "
          f"{shutil.disk_usage(WORK).free/1e9:5.1f} GB free on {WORK}   {tag}")


def _free(p):
    try:
        return shutil.disk_usage(p).free
    except OSError:
        return -1


# predictions.txt runs to several hundred MB and /kaggle/working is the tightest mount,
# so the big intermediate goes wherever there is actually room; only the zip must land
# in /kaggle/working to be downloadable.
SCRATCH = max([WORK, Path("/tmp"), Path("/kaggle/temp")], key=_free)


print("polars", pl.__version__)
print(f"RAM available to this session: {psutil.virtual_memory().total/1e9:.0f} GB")
for _p in (WORK, Path("/tmp"), Path("/kaggle/temp")):
    if _free(_p) >= 0:
        print(f"   {str(_p):<16} {_free(_p)/1e9:6.1f} GB free")
print(f"scratch for large intermediates: {SCRATCH}")

# /kaggle/working is a small mount that survives between runs of the same notebook, and
# this notebook used to write ~1.5 GB into it. Clear only the files it creates itself,
# then list whatever else is in there -- a full working dir is what ENOSPC actually means.
for _p in ("contrastive.zip", "predictions.txt", "ebnerd_predictions.zip"):
    if (WORK / _p).exists():
        print(f"   removing stale {_p} ({(WORK / _p).stat().st_size/1e6:.0f} MB)")
        (WORK / _p).unlink()
if (WORK / "Ekstra_Bladet_contrastive_vector").exists():
    print("   removing stale Ekstra_Bladet_contrastive_vector/")
    shutil.rmtree(WORK / "Ekstra_Bladet_contrastive_vector", ignore_errors=True)

_left = [q for q in WORK.rglob("*") if q.is_file()]
print(f"{WORK} now holds {sum(q.stat().st_size for q in _left)/1e9:.2f} GB "
      f"in {len(_left)} files, {_free(WORK)/1e9:.1f} GB free")
for q in sorted(_left, key=lambda x: -x.stat().st_size)[:8]:
    print(f"   {q.stat().st_size/1e6:9.1f} MB  {q.relative_to(WORK)}")
mem("startup")


def find_dir(*must_contain):
    """Locate a directory under /kaggle/input holding all the named files."""
    for cand in glob.glob("/kaggle/input/**/", recursive=True):
        if all((Path(cand) / f).exists() for f in must_contain):
            return Path(cand)
    raise FileNotFoundError(f"no directory contains {must_contain}")


TRAIN_DIR = find_dir("behaviors.parquet", "history.parquet")
LARGE = None
for cand in glob.glob("/kaggle/input/**/ebnerd_large/", recursive=True):
    LARGE = Path(cand); break
TEST = None
for cand in glob.glob("/kaggle/input/**/ebnerd_testset/", recursive=True):
    if (Path(cand) / "test").exists():
        TEST = Path(cand); break
    inner = Path(cand) / "ebnerd_testset"
    if (inner / "test").exists():
        TEST = inner; break

assert LARGE is not None, "could not find ebnerd_large/"
assert TEST is not None, "could not find ebnerd_testset/ with a test/ subfolder"
print("large:", LARGE)
print("test: ", TEST)
for p in [LARGE / "articles.parquet", LARGE / "train/behaviors.parquet",
          LARGE / "validation/behaviors.parquet", TEST / "test/behaviors.parquet",
          TEST / "test/history.parquet"]:
    print(f"   {'OK ' if p.exists() else 'MISSING'} {p.name:<20} {p.parent.name}")

polars 1.35.2
RAM available to this session: 34 GB
   /kaggle/working    20.9 GB free
   /tmp             1100.0 GB free
scratch for large intermediates: /tmp
/kaggle/working now holds 0.00 GB in 1 files, 20.9 GB free
         0.1 MB  .virtual_documents/__notebook_source__.ipynb
   [mem] 11.63 GB |  20.9 GB free on /kaggle/working   startup
large: /kaggle/input/datasets/suyashnpande/ebnerd-large-dataset2/ebnerd_large
test:  /kaggle/input/datasets/suyashnpande/ebnerd-large-dataset2/ebnerd_testset/ebnerd_testset
   OK  articles.parquet     ebnerd_large
   OK  behaviors.parquet    train
   OK  behaviors.parquet    validation
   OK  behaviors.parquet    test
   OK  history.parquet      test


### The article embeddings

Ekstra Bladet's own contrastively-trained model, built on this exact Danish corpus. The
bucket is public, so we pull it straight in rather than uploading anything — and cache it
to `/kaggle/working` so a re-run inside the same session is instant.

In [24]:
# Robust download + validation of the EB-NeRD contrastive embedding artifact.
# A previous run can leave a partial/HTML response at contrastive.zip; never pass
# an unvalidated file to zipfile.ZipFile.
ART_URL = "https://ebnerd-dataset.s3.eu-west-1.amazonaws.com/artifacts/Ekstra_Bladet_contrastive_vector.zip"
ART_ZIP = SCRATCH / "contrastive.zip"
ART_PART = SCRATCH / "contrastive.zip.part"

def valid_zip(path):
    try:
        return path.exists() and path.stat().st_size > 1_000_000 and zipfile.is_zipfile(path)
    except OSError:
        return False

# Remove stale/corrupt downloads from previous runs.
if ART_ZIP.exists() and not valid_zip(ART_ZIP):
    print(f"removing invalid contrastive archive: {ART_ZIP} ({ART_ZIP.stat().st_size/1e6:.1f} MB)")
    ART_ZIP.unlink()
ART_PART.unlink(missing_ok=True)

if not valid_zip(ART_ZIP):
    t0 = time.time()
    print("downloading the EB-NeRD contrastive embedding artifact (~341 MB) ...")
    req = urllib.request.Request(
        ART_URL,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as response, open(ART_PART, "wb") as out:
            shutil.copyfileobj(response, out, length=1024*1024)
        if not valid_zip(ART_PART):
            head = ART_PART.read_bytes()[:200]
            raise RuntimeError(
                f"Downloaded file is not a valid ZIP. size={ART_PART.stat().st_size/1e6:.1f} MB; "
                f"first bytes={head!r}. Enable Kaggle Internet and rerun this cell."
            )
        ART_PART.replace(ART_ZIP)
        print(f"   done in {time.time()-t0:.0f}s ({ART_ZIP.stat().st_size/1e6:.0f} MB)")
    except Exception:
        ART_PART.unlink(missing_ok=True)
        raise
else:
    print(f"using existing valid archive: {ART_ZIP} ({ART_ZIP.stat().st_size/1e6:.0f} MB)")

with zipfile.ZipFile(ART_ZIP, "r") as z:
    members = [n for n in z.namelist()
               if n.endswith(".parquet") and not n.startswith("__MACOSX") and not n.endswith("/")]
    assert members, "contrastive archive contains no parquet file"
    print("archive parquet files:", members)
    z.extractall(SCRATCH, members=members)

EMB_PATH = SCRATCH / members[0]
assert EMB_PATH.exists(), f"extracted embedding parquet not found: {EMB_PATH}"
emb = pl.read_parquet(EMB_PATH)
print(f"\nembedding table: {emb.height:,} rows, columns {emb.columns}")
VEC_COL = [c for c in emb.columns if c != "article_id"][0]
DIM = len(emb[VEC_COL][0])
print(f"vector column '{VEC_COL}', dimension {DIM}")


removing invalid contrastive archive: /tmp/contrastive.zip (222.5 MB)
downloading the EB-NeRD contrastive embedding artifact (~341 MB) ...
   done in 14s (358 MB)
archive parquet files: ['Ekstra_Bladet_contrastive_vector/contrastive_vector.parquet']

embedding table: 125,541 rows, columns ['article_id', 'contrastive_vector']
vector column 'contrastive_vector', dimension 768


## 2. Metrics — official definitions plus a verified fast path

In [25]:
from sklearn.metrics import roc_auc_score


def dcg_score(y_true, y_score, k=10):
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order[:k])
    return np.sum((2 ** y_true - 1) / np.log2(np.arange(len(y_true)) + 2))


def ndcg_score(y_true, y_score, k=10):
    return dcg_score(y_true, y_score, k) / dcg_score(y_true, y_true, k)


def mrr_score(y_true, y_score):
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order)
    return np.sum(y_true / (np.arange(len(y_true)) + 1)) / np.sum(y_true)


def score_official(L, S):
    a, m, n5, n10 = [], [], [], []
    for y, s in zip(L, S):
        y = np.asarray(y, dtype="float32")
        if y.sum() == 0 or y.sum() == len(y):
            continue
        s = np.asarray(s, dtype="float64")
        a.append(roc_auc_score(y, s)); m.append(mrr_score(y, s))
        n5.append(ndcg_score(y, s, 5)); n10.append(ndcg_score(y, s, 10))
    return {"AUC": np.mean(a), "MRR": np.mean(m), "nDCG@5": np.mean(n5),
            "nDCG@10": np.mean(n10), "n": len(a)}


def score_fast(imp_ids, labels, scores):
    """Every metric for every impression at once, via group-wise ranking."""
    df = pl.DataFrame({"imp": imp_ids, "y": np.asarray(labels, dtype=np.float64),
                       "s": np.asarray(scores, dtype=np.float64)})
    df = df.with_columns([
        pl.col("s").rank(method="average").over("imp").alias("r_asc"),
        pl.col("s").rank(method="ordinal", descending=True).over("imp").alias("r_desc"),
    ]).with_columns([
        (pl.col("y") * pl.col("r_asc")).alias("pos_rank"),
        (pl.col("y") / pl.col("r_desc")).alias("rr"),
        pl.when(pl.col("r_desc") <= 5).then(pl.col("y") / (pl.col("r_desc") + 1).log(2))
          .otherwise(0.0).alias("g5"),
        pl.when(pl.col("r_desc") <= 10).then(pl.col("y") / (pl.col("r_desc") + 1).log(2))
          .otherwise(0.0).alias("g10"),
    ])
    g = df.group_by("imp").agg([
        pl.len().alias("n"), pl.col("y").sum().alias("npos"),
        pl.col("pos_rank").sum().alias("rank_sum"), pl.col("rr").sum().alias("rr_sum"),
        pl.col("g5").sum().alias("dcg5"), pl.col("g10").sum().alias("dcg10"),
    ]).filter((pl.col("npos") > 0) & (pl.col("npos") < pl.col("n")))
    n = g["n"].to_numpy(); npos = g["npos"].to_numpy()
    auc = (g["rank_sum"].to_numpy() - npos * (npos + 1) / 2) / (npos * (n - npos))
    disc = 1.0 / np.log2(np.arange(2, 12))
    cum = np.concatenate([[0.0], np.cumsum(disc)])
    return {"AUC": auc.mean(), "MRR": (g["rr_sum"].to_numpy() / npos).mean(),
            "nDCG@5": (g["dcg5"].to_numpy() / cum[np.minimum(npos, 5).astype(int)]).mean(),
            "nDCG@10": (g["dcg10"].to_numpy() / cum[np.minimum(npos, 10).astype(int)]).mean(),
            "n": len(auc)}


# Verify before relying on it. AUC uses average ranks so it is tie-safe and must match
# exactly; MRR/nDCG need a strict order and the official np.argsort(...)[::-1] is not a
# stable descending sort, so under ties its ordering is arbitrary.
rng = np.random.default_rng(0)
imp, ys, ss, L, S = [], [], [], [], []
for i in range(4000):
    k = int(rng.integers(2, 25))
    y = np.zeros(k, dtype=int); y[rng.integers(0, k)] = 1     # EB-NeRD: ~1 click/impression
    s = np.round(rng.normal(size=k), 2)
    imp += [i] * k; ys += list(y); ss += list(s); L.append(y); S.append(s)
f, o = score_fast(np.array(imp), np.array(ys), np.array(ss)), score_official(L, S)
for k in ["AUC", "MRR", "nDCG@5", "nDCG@10"]:
    print(f"   {k:<9} fast {f[k]:.6f}   official {o[k]:.6f}   diff {abs(f[k]-o[k]):.2e}")
assert abs(f["AUC"] - o["AUC"]) < 1e-9, "AUC must match exactly"
assert max(abs(f[k] - o[k]) for k in ["MRR", "nDCG@5", "nDCG@10"]) < 1e-3
print("\nverified against the official definitions")

   AUC       fast 0.499516   official 0.499516   diff 0.00e+00
   MRR       fast 0.300269   official 0.300344   diff 7.55e-05
   nDCG@5    fast 0.308164   official 0.308057   diff 1.08e-04
   nDCG@10   fast 0.394901   official 0.395029   diff 1.28e-04

verified against the official definitions


## 3. Load

`history.parquet` is 1.2 GB and holds one list per user covering 21 days. We project to
just the two columns we need and truncate each list to its most recent entries **during**
the scan, so the full column never lands in memory.

In [26]:
HIST_CAP = 30          # most recent clicks kept per user

# Project at read time. The full behaviours schema also carries read_time,
# scroll_percentage, device_type, demographics and the next_* columns, none of which we
# use; at 12M+ rows per split that is several GB of nothing. Ask parquet for five columns.
BH_COLS = ["impression_id", "impression_time", "user_id",
           "article_ids_inview", "article_ids_clicked", "session_id"]
# articles.parquet is 82% `body` (full article text) plus url/ner_clusters/topics/
# image_ids, none of which we read. Nine columns instead of seventeen.
ART_COLS = ["article_id", "published_time", "category_str", "subcategory",
            "total_pageviews", "total_inviews", "sentiment_score", "title", "subtitle"]

def read_history(path):
    return (pl.scan_parquet(path)
              .select([pl.col("user_id"),
                       pl.col("article_id_fixed").list.tail(HIST_CAP).alias("hist")])
              .collect(streaming=True))


t0 = time.time()
articles = pl.read_parquet(LARGE / "articles.parquet", columns=ART_COLS)
bh_train = pl.read_parquet(LARGE / "train/behaviors.parquet", columns=BH_COLS)
bh_val = pl.read_parquet(LARGE / "validation/behaviors.parquet", columns=BH_COLS)
bh_test = pl.read_parquet(TEST / "test/behaviors.parquet",
                          columns=[c for c in BH_COLS if c != "article_ids_clicked"])
print(f"behaviors loaded in {time.time()-t0:.0f}s")

# Explicit split boundaries used by leakage/correctness checks.
# Derive them from the actual behavior tables rather than relying on undeclared constants.
TRAIN_START = bh_train["impression_time"].min()
VAL_START = bh_val["impression_time"].min()
TEST_START = bh_test["impression_time"].min()
print(f"split starts: train={TRAIN_START} | validation={VAL_START} | test={TEST_START}")

# Each split ships its own 21-day lookback, ending exactly where that split's behaviours
# begin. Pooling them and keeping one row per user takes the LAST file -- the test
# history, which runs to the start of the test week and therefore covers the whole
# validation week we tune on. That put the labels being predicted inside the user
# profile. Keep them separate: validation history to evaluate and tune, test history to
# predict. Same code path, different lookback.
t0 = time.time()
HIST_TRAIN = read_history(LARGE / "train/history.parquet")
HIST_EVAL = read_history(LARGE / "validation/history.parquet")
HIST_PRED = read_history(TEST / "test/history.parquet")
print(f"history: {HIST_EVAL.height:,} eval users | {HIST_PRED.height:,} pred users "
      f"in {time.time()-t0:.0f}s")

print(f"\ntrain {bh_train.height:,} | validation {bh_val.height:,} | test {bh_test.height:,}")
print(f"articles {articles.height:,}")
TEST_SCHEMA = pl.scan_parquet(TEST / "test/behaviors.parquet").collect_schema().names()
print("\ntest columns:", TEST_SCHEMA)
HAS_BA = "is_beyond_accuracy" in TEST_SCHEMA
print(f"is_beyond_accuracy present: {HAS_BA}")
for nm, df in [("train", bh_train), ("validation", bh_val), ("test", bh_test)]:
    print(f"   {nm:<11} {df['impression_time'].min()}  ->  {df['impression_time'].max()}")

TEST_ROWS = int(bh_test["article_ids_inview"].list.len().sum())
print(f"\ntest candidate rows: {TEST_ROWS:,} "
      f"({TEST_ROWS/bh_test.height:.1f} per impression) <- what section 10 has to score")

behaviors loaded in 8s
split starts: train=2023-05-18 07:00:00 | validation=2023-05-25 07:00:00 | test=2023-06-01 07:00:00
history: 791,582 eval users | 807,677 pred users in 6s

train 12,063,890 | validation 12,566,385 | test 13,536,710
articles 125,541

test columns: ['impression_id', 'impression_time', 'read_time', 'scroll_percentage', 'device_type', 'article_ids_inview', 'user_id', 'is_sso_user', 'gender', 'postcode', 'age', 'is_subscriber', 'session_id', 'is_beyond_accuracy']
is_beyond_accuracy present: True
   train       2023-05-18 07:00:00  ->  2023-05-25 06:59:59
   validation  2023-05-25 07:00:00  ->  2023-06-01 06:59:59
   test        2023-06-01 07:00:00  ->  2023-06-08 06:59:59

test candidate rows: 205,925,868 (15.2 per impression) <- what section 10 has to score


### Session context (serving-time)

EB-NeRD exposes `session_id`. We add a causal session-depth feature: the zero-based position of the current impression within its session. It is computed independently inside each split after sorting by impression time, so no future session impressions are used. `read_time`/`scroll_percentage` are intentionally not used in the final ranker because they are post-impression outcomes rather than features known when the recommendation is served.


In [27]:
# ============================================================
# Session-depth maps, with no future rows
# ============================================================
def build_session_depth(df):
    if "session_id" not in df.columns:
        return {}
    x=df.select(["impression_id","session_id","impression_time"]).sort("impression_time")
    x=x.with_columns(pl.int_range(pl.len()).over("session_id").alias("session_depth"))
    return {int(i):int(d) for i,d in zip(x["impression_id"].to_list(),x["session_depth"].to_list())}
SESSION_DEPTH_TRAIN=build_session_depth(bh_train)
SESSION_DEPTH_EVAL=build_session_depth(bh_val)
SESSION_DEPTH_PRED=build_session_depth(bh_test)
print("session-depth maps:",len(SESSION_DEPTH_TRAIN),len(SESSION_DEPTH_EVAL),len(SESSION_DEPTH_PRED))


session-depth maps: 12063890 12566385 13336711


## 4. Article features

Four signals, three of which MIND simply did not have.

`total_pageviews` is the strongest single feature we measured (AUC 0.5969 alone). One
caveat we record rather than hide: it counts clicks in the **first 7 days after
publication**, so for an article published during the test week it partly reflects the
future. It is supplied in the dataset and therefore fair game for the leaderboard, but it
is not something a live system could read at serving time — exactly the Q9
"features unavailable at serving time" category, and the ablation below reports the score
with and without it.

In [28]:
# Age buckets measured on the demo bundle: click rate peaks at 2-4h and falls away on
# BOTH sides, so a linear term cannot express it (it scored 0.4950, below random).
AGE_EDGES = np.array([1, 2, 4, 8, 12, 18, 24, 48, 120], dtype=np.float64)
AGE_VALUE = np.array([0.0773, 0.1090, 0.1193, 0.1112, 0.1108,
                      0.0884, 0.0764, 0.0445, 0.0394, 0.0146])

ART = articles.select([
    pl.col("article_id"),
    pl.col("published_time"),
    pl.col("category_str").fill_null("").alias("category"),
    pl.col("subcategory").list.first().fill_null(-1).alias("subcat"),
    pl.col("total_pageviews").fill_null(0).alias("pageviews"),
    pl.col("total_inviews").fill_null(0).alias("inviews"),
    pl.col("sentiment_score").fill_null(0.0).alias("sentiment"),
    pl.col("title").fill_null(""),
    pl.col("subtitle").fill_null(""),
]).sort("article_id")

CATALOGUE = ART["article_id"].to_list()
IDX = {int(a): i for i, a in enumerate(CATALOGUE)}
ART_IDS = np.asarray(CATALOGUE, dtype=np.int64)    # sorted: ART was sorted by article_id
ID_DTYPE = ART.schema["article_id"]
print(f"catalogue {len(CATALOGUE):,} articles")


def lookup(sorted_keys, values):
    """Vectorised dict.get over a sorted key array -> (positions, found mask)."""
    v = np.asarray(values, dtype=np.int64)
    p = np.searchsorted(sorted_keys, v)
    np.clip(p, 0, len(sorted_keys) - 1, out=p)
    return p, sorted_keys[p] == v

# log1p because pageviews spans several orders of magnitude
ART = ART.with_columns([
    (pl.col("pageviews") + 1).log().alias("log_pv"),
    (pl.col("inviews") + 1).log().alias("log_iv"),
])
print(f"pageviews non-zero on {100*(ART['pageviews']>0).mean():.1f}% of articles")

# Embeddings, re-indexed onto the catalogue order so a row index means the same thing
# everywhere. Articles without a vector keep zeros, which score 0 against everything.
VEC = np.zeros((len(CATALOGUE), DIM), dtype=np.float32)
src = {int(a): i for i, a in enumerate(emb["article_id"].to_list())}
# explode to one flat buffer rather than np.stack over 125k little object arrays
raw = (emb[VEC_COL].explode().to_numpy()
       .astype(np.float32, copy=False).reshape(emb.height, DIM))
missing = 0
for a, i in IDX.items():
    j = src.get(a)
    if j is None:
        missing += 1
    else:
        VEC[i] = raw[j]
n = np.linalg.norm(VEC, axis=1, keepdims=True)
VEC /= np.maximum(n, 1e-12)                    # unit vectors => dot product is cosine
del raw, src, emb
# ~716 MB of zip plus extracted parquet, now fully absorbed into VEC
shutil.rmtree(SCRATCH / "Ekstra_Bladet_contrastive_vector", ignore_errors=True)
ART_ZIP.unlink(missing_ok=True)
gc.collect()
print(f"embeddings: {missing:,} of {len(CATALOGUE):,} articles without a vector "
      f"({100*missing/len(CATALOGUE):.2f}%), matrix {VEC.nbytes/1e6:.0f} MB")
mem("after embeddings")

catalogue 125,541 articles
pageviews non-zero on 13.5% of articles
embeddings: 0 of 125,541 articles without a vector (0.00%), matrix 386 MB
   [mem] 13.41 GB |  20.9 GB free on /kaggle/working   after embeddings


## 5. BM25 over Danish text

Same inverted index as MIND, but with Danish stopwords. Danish is a compounding language —
`fodboldlandsholdet` is one token meaning "the national football team" — so exact term
matching is structurally handicapped here. We include BM25 anyway because it completes Q2
at scale and because the MIND run showed it earns a non-zero weight even when semantic
dominates.

In [29]:
TOKEN = re.compile(r"[^\W_]+", re.UNICODE)
STOP_DA = set(("og i jeg det at en den til er som på de med han af for ikke der var mig "
               "sig men et har om vi min havde ham hun nu over da fra du ud sin dem os "
               "op man hans hvor eller hvad skal selv her alle vil blev kunne ind når "
               "være dog no mit efter" ).split())
K1, B_ = 1.2, 0.75


def tok(s):
    return [w for w in TOKEN.findall(s.lower()) if len(w) > 2 and w not in STOP_DA]


t0 = time.time()
docs = [tok(t + " " + s) for t, s in zip(ART["title"].to_list(), ART["subtitle"].to_list())]
vocab, postings = {}, []
doc_len = np.zeros(len(docs), dtype=np.float32)
for d, ts in enumerate(docs):
    doc_len[d] = len(ts)
    cnt = defaultdict(int)
    for w in ts:
        t = vocab.get(w)
        if t is None:
            t = len(vocab); vocab[w] = t; postings.append({})
        cnt[t] += 1
    for t, tf in cnt.items():
        postings[t][d] = tf

avgdl = float(doc_len.mean()); N = len(docs)
nnz = sum(len(p) for p in postings)
rows = np.empty(nnz, np.int32); cols = np.empty(nnz, np.int32)
data = np.empty(nnz, np.float32); tfs = np.empty(nnz, np.float32)
pos = 0
for t, p in enumerate(postings):
    if not p:
        continue
    idf = np.log(1.0 + (N - len(p) + 0.5) / (len(p) + 0.5))
    for d, tf in p.items():
        norm = K1 * (1.0 - B_ + B_ * doc_len[d] / avgdl)
        rows[pos] = d; cols[pos] = t; data[pos] = idf * tf * (K1 + 1.0) / (tf + norm)
        tfs[pos] = tf
        pos += 1
BM25_W = sp.csr_matrix((data[:pos], (rows[:pos], cols[:pos])), shape=(N, len(vocab)))
# The raw counts as well. A user's BM25 query is the summed term counts of the articles
# they clicked, which is exactly (click matrix @ TF) -- see section 7.
TF = sp.csr_matrix((tfs[:pos], (rows[:pos], cols[:pos])), shape=(N, len(vocab)))
del postings, rows, cols, data, tfs, docs
gc.collect()
print(f"BM25: {N:,} docs, {len(vocab):,} terms, {BM25_W.nnz:,} postings "
      f"in {time.time()-t0:.0f}s")

BM25: 125,541 docs, 119,851 terms, 1,774,642 postings in 8s


## 6. Click-through rate, built to a cutoff

Same idea that worked on MIND: an exponential half-life so recent clicks count for more,
and the ability to build the table to an arbitrary cutoff so we can tune under the same
staleness the test set will have.

In [30]:
HALF_LIFE_H = 24.0
ALPHA = 20.0
CTR_STEP = 500_000
# A list of projections rather than pl.concat: selecting columns is a refcount bump on the
# Arrow buffers, whereas concatenating 24.6M rows would copy ~1.5 GB for no benefit.
_LAB_COLS = ["impression_time", "article_ids_inview", "article_ids_clicked"]
LABELLED = [bh_train.select(_LAB_COLS), bh_val.select(_LAB_COLS)]


def build_ctr(cutoff):
    """Decay-weighted click-through rate per article, accumulated in fixed-size slices.

    Exploding every labelled impression at once is ~296M rows and several GB in a single
    intermediate, and this runs three times. Accumulating into two arrays over slices
    keeps the transient flat no matter how much history the cutoff admits.
    """
    assert cutoff is not None, "null cutoff: check how it was derived"
    parts = [p.filter(pl.col("impression_time") < cutoff) for p in LABELLED]
    parts = [p for p in parts if p.height]
    assert parts, f"no labelled impressions before {cutoff}"
    ref = max(p["impression_time"].max() for p in parts)

    shown = np.zeros(len(CATALOGUE), np.float64)
    clicks = np.zeros(len(CATALOGUE), np.float64)
    for part in parts:
        for st in range(0, part.height, CTR_STEP):
            ex = (part.slice(st, CTR_STEP)
                      .with_columns(pl.col("article_ids_inview").alias("a")).explode("a")
                      .with_columns([
                          pl.col("a").is_in(pl.col("article_ids_clicked"))
                            .cast(pl.Int8).alias("y"),
                          (0.5 ** (((pl.lit(ref) - pl.col("impression_time"))
                                    .dt.total_seconds() / 3600.0) / HALF_LIFE_H)).alias("w"),
                      ]).select(["a", "y", "w"]))
            ai, hit = lookup(ART_IDS, ex["a"].to_numpy())
            w = ex["w"].to_numpy()[hit]
            ai = ai[hit]
            shown += np.bincount(ai, weights=w, minlength=len(CATALOGUE))
            clicks += np.bincount(ai, weights=w * ex["y"].to_numpy()[hit],
                                  minlength=len(CATALOGUE))
            del ex
    g = float(clicks.sum() / max(shown.sum(), 1e-12))
    keep = shown > 0
    return pl.DataFrame({
        "article_id": ART_IDS[keep],
        "ctr": (clicks[keep] + ALPHA * g) / (shown[keep] + ALPHA),
    }).with_columns(pl.col("article_id").cast(ID_DTYPE)), g


# The test week follows validation, so "CTR from train, evaluated on validation" already
# reproduces the staleness the test set will see. We build a second, staler table to show
# the effect explicitly and to tune on the harder of the two.
# NB: Series.quantile() on a datetime column returns null on polars < ~1.40 (Kaggle ships
# 1.35), which silently makes the cutoff null, empties the filter, and only surfaces much
# later as "- not allowed on null and datetime". Plain datetime arithmetic instead: it is
# the true temporal midpoint and behaves the same on every version.
# VAL_START was defined from bh_val immediately after loading the split tables.
VAL_MID = VAL_START + (bh_val["impression_time"].max() - VAL_START) / 2
print(f"validation midpoint: {VAL_MID}")
t0 = time.time()
CTR_FRESH, G_FRESH = build_ctr(VAL_MID)                 # includes half of validation
CTR_STALE, G_STALE = build_ctr(VAL_START)               # train only
CTR_FINAL, G_FINAL = build_ctr(bh_test["impression_time"].min())   # everything labelled
print(f"fresh {CTR_FRESH.height:,} | stale {CTR_STALE.height:,} | final {CTR_FINAL.height:,} "
      f"articles ({time.time()-t0:.0f}s)")
mem("after CTR tables")

validation midpoint: 2023-05-28 18:59:59.500000
fresh 11,474 | stale 10,116 | final 12,818 articles (170s)
   [mem] 15.05 GB |  20.9 GB free on /kaggle/working   after CTR tables


### Recalibrating the age curve

`AGE_VALUE` in section 4 was measured on the demo bundle, roughly 1% of this corpus. Now
that the real logs are loaded we recompute it here rather than transplanting constants
across a 100x scale gap, and print both so the difference is on the record. Buckets with
too little support keep the demo value.

In [31]:
MIN_BUCKET = 5_000
AGE_LABELS = ["<1h", "1-2h", "2-4h", "4-8h", "8-12h", "12-18h", "18-24h", "24-48h",
              "48-120h", ">120h"]

t0 = time.time()
# Train only -- fitting this on validation would leak into the tuning we do there. And
# strided rather than head(), so the sample spans the whole training week instead of its
# first few hours, when every article in the catalogue still looks freshly published.
_step = max(1, LABELLED[0].height // 1_000_000)
_src = (LABELLED[0].gather_every(_step)
        .with_columns(pl.col("article_ids_inview").alias("a")).explode("a")
        .with_columns(pl.col("a").is_in(pl.col("article_ids_clicked")).cast(pl.Int8).alias("y"))
        .join(ART.select(["article_id", "published_time"]),
              left_on="a", right_on="article_id", how="left")
        .with_columns(((pl.col("impression_time") - pl.col("published_time"))
                       .dt.total_seconds() / 3600.0).alias("age_h"))
        .drop_nulls("age_h").filter(pl.col("age_h") >= 0))
_g = (_src.with_columns(pl.Series("b", np.searchsorted(AGE_EDGES, _src["age_h"].to_numpy())))
          .group_by("b").agg([pl.len().alias("n"), pl.col("y").mean().alias("ctr")])
          .sort("b"))

AGE_DEMO = AGE_VALUE.copy()
AGE_VALUE = AGE_VALUE.copy()
_seen = dict(zip(_g["b"].to_list(), zip(_g["n"].to_list(), _g["ctr"].to_list())))
for b, (cnt, ctr) in _seen.items():
    if cnt >= MIN_BUCKET:
        AGE_VALUE[b] = ctr

print(f"recomputed on {_src.height:,} candidate rows ({time.time()-t0:.0f}s)\n")
print(f"{'bucket':<9} {'rows':>12} {'demo':>8} {'large':>8}")
for i, lab in enumerate(AGE_LABELS):
    cnt = _seen.get(i, (0, 0.0))[0]
    flag = "" if cnt >= MIN_BUCKET else "   <- too few, kept demo"
    print(f"{lab:<9} {cnt:>12,} {AGE_DEMO[i]:>8.4f} {AGE_VALUE[i]:>8.4f}{flag}")
print(f"\nspread (max/min): demo {AGE_DEMO.max()/AGE_DEMO.min():.1f}x  "
      f"large {AGE_VALUE.max()/AGE_VALUE.min():.1f}x")
print(f"peak bucket: demo {AGE_LABELS[int(AGE_DEMO.argmax())]}, "
      f"large {AGE_LABELS[int(AGE_VALUE.argmax())]}")
del _src, _g
# LABELLED and bh_train have now done their only jobs -- the CTR tables above and this
# age curve. Together they are ~2 GB that nothing downstream reads. pop() rather than del
# so re-running the cell is not a NameError.
# Keep bh_train and HIST_TRAIN for the A2 NRMS training stage.
# LABELLED is no longer needed.
globals().pop("LABELLED", None)
gc.collect()
mem("after freeing LABELLED; bh_train retained for NRMS")

recomputed on 11,140,470 candidate rows (5s)

bucket            rows     demo    large
<1h          1,866,281   0.0773   0.0838
1-2h         1,779,294   0.1090   0.1161
2-4h         2,005,446   0.1193   0.1263
4-8h         1,421,123   0.1112   0.1292
8-12h          857,426   0.1108   0.1319
12-18h         365,322   0.0884   0.0975
18-24h         173,531   0.0764   0.0497
24-48h         475,538   0.0445   0.0334
48-120h        324,340   0.0394   0.0288
>120h        1,872,169   0.0146   0.0141

spread (max/min): demo 8.2x  large 9.4x
peak bucket: demo 2-4h, large 8-12h
   [mem] 16.98 GB |  20.9 GB free on /kaggle/working   after freeing LABELLED; bh_train retained for NRMS


## 7. Feature frame

In [32]:
ROW_CHUNK = 100_000


# ------------------------------------------------- the click matrix, and nothing derived
# Everything a user contributes -- their embedding, their BM25 query, their category mix --
# is a product against H, the user x article click matrix. H is small (~0.15 GB each). The
# products are not: materialising them for every user costs GBs, and we only ever look at
# one batch at a time, so each batch rebuilds its own slice as a sparse matmul.
#
# Two lookbacks, never pooled. Each split ships its own 21 days ending exactly where that
# split's behaviours begin; the test history runs to the start of the test week and so
# covers the whole validation week we tune on. Keeping them apart is what stops the labels
# being predicted from appearing inside the user profile.
t0 = time.time()


def click_matrix(hist):
    """(sorted user ids, user x article click matrix) for one history lookback."""
    hist = hist.sort("user_id")             # so a user's row index is searchsortable
    uid = hist["user_id"].to_numpy().astype(np.int64)
    hx = hist.with_row_index("urow").select(["urow", "hist"]).explode("hist").drop_nulls()
    ai, hit = lookup(ART_IDS, hx["hist"].to_numpy())
    H = sp.csr_matrix((np.ones(int(hit.sum()), np.float32),
                       (hx["urow"].to_numpy()[hit], ai[hit])),
                      shape=(len(uid), len(CATALOGUE)))
    return uid, H


UID_TRAIN, H_TRAIN = click_matrix(HIST_TRAIN)
UID_EVAL, H_EVAL = click_matrix(HIST_EVAL)
UID_PRED, H_PRED = click_matrix(HIST_PRED)
gc.collect()
print(f"click matrices in {time.time()-t0:.0f}s")
for _nm, _u, _H in [("train (training lookback)", UID_TRAIN, H_TRAIN), ("eval (validation lookback)", UID_EVAL, H_EVAL),
                    ("pred (test lookback)      ", UID_PRED, H_PRED)]:
    print(f"   {_nm}: {len(_u):,} users, {_H.nnz:,} clicks, "
          f"{(_H.data.nbytes + _H.indices.nbytes)/1e9:.2f} GB")


def onehot(codes, width):
    return sp.csr_matrix((np.ones(len(codes), np.float32),
                          (np.arange(len(codes)), codes)), shape=(len(codes), width))


# ------------------------------------------------- article features as flat arrays
# These are constant across every batch, so joining them onto 300k rows 687 times was
# pure repetition. Held as numpy arrays indexed by catalogue position and gathered with
# the candidate index instead, which removes both hash joins from the hot loop.
A_LOGPV = ART["log_pv"].to_numpy().astype(np.float32)
A_LOGIV = ART["log_iv"].to_numpy().astype(np.float32)
A_SENT = ART["sentiment"].to_numpy().astype(np.float32)
_pub = ART["published_time"].to_numpy()
A_PUB_OK = ~np.isnat(_pub)
A_PUB = np.where(A_PUB_OK, _pub.astype("datetime64[s]").astype(np.int64), 0)
del _pub

_c = ART["category"].to_list()
_cmap = {v: i for i, v in enumerate(sorted(set(_c)))}
CAT_CODE = np.fromiter((_cmap[v] for v in _c), np.int32, len(_c))
_s = ART["subcat"].to_list()
_smap = {v: i for i, v in enumerate(sorted(set(_s)))}
SUB_CODE = np.fromiter((_smap[v] for v in _s), np.int32, len(_s))
C_CAT, C_SUB = onehot(CAT_CODE, len(_cmap)), onehot(SUB_CODE, len(_smap))
del _c, _s


def ctr_vector(tbl, g):
    """A CTR table spread onto catalogue positions, with the global rate as the default."""
    v = np.full(len(CATALOGUE), g, dtype=np.float32)
    p, hit = lookup(ART_IDS, tbl["article_id"].to_numpy())
    v[p[hit]] = tbl["ctr"].to_numpy()[hit].astype(np.float32)
    return v


CTRV_FRESH = ctr_vector(CTR_FRESH, G_FRESH)
CTRV_STALE = ctr_vector(CTR_STALE, G_STALE)
CTRV_FINAL = ctr_vector(CTR_FINAL, G_FINAL)


def prepare(df, ctr_vec, labelled):
    """Explode a behaviours slice to candidate rows, as plain numpy arrays.

    Rows come out contiguous per impression and in the original order, which is what the
    scorer requires and what lets ranking be a segmented sort rather than a group-by.
    """
    cols = ["impression_id", "user_id", "impression_time", "article_ids_inview"]
    if labelled:
        cols.append("article_ids_clicked")
    b = df.select(cols)
    imp_ids = b["impression_id"].to_numpy()
    lens = b["article_ids_inview"].list.len().to_numpy().astype(np.int64)
    ex = b.explode("article_ids_inview").rename({"article_ids_inview": "article_id"})
    if labelled:
        label = (ex.select(pl.col("article_id").is_in(pl.col("article_ids_clicked"))
                             .cast(pl.Int8).alias("y"))["y"].to_numpy())
    else:
        label = np.zeros(ex.height, dtype=np.int8)

    ci, c_ok = lookup(ART_IDS, ex["article_id"].to_numpy())
    ts = ex["impression_time"].to_numpy().astype("datetime64[s]").astype(np.int64)
    have_age = c_ok & A_PUB_OK[ci]
    age = np.where(have_age, np.maximum((ts - A_PUB[ci]) / 3600.0, 0.0), 0.0)
    z = np.zeros(ex.height, dtype=np.float32)
    return {
        "imp": np.repeat(imp_ids, lens), "imp_ids": imp_ids, "lens": lens,
        "label": label, "ci": ci, "c_ok": c_ok,
        "users": ex["user_id"].to_numpy(), "n": ex.height,
        "age_score": AGE_VALUE[np.searchsorted(AGE_EDGES, age)],
        "ctr": np.where(c_ok, ctr_vec[ci], ctr_vec.mean()),
        "pv": np.where(c_ok, A_LOGPV[ci], z), "iv": np.where(c_ok, A_LOGIV[ci], z),
        "sent": np.where(c_ok, A_SENT[ci], z),
    }


def zscore(x):
    x = np.asarray(x, dtype=np.float64); sd = x.std()
    return (x - x.mean()) / sd if sd > 1e-12 else np.zeros_like(x)


def components(F, uid, H, row_chunk=ROW_CHUNK):
    """Every scoring signal for one prepared batch.

    `uid`/`H` name the history lookback explicitly -- UID_EVAL/H_EVAL when scoring
    validation, UID_PRED/H_PRED when predicting the test week. Passing it rather than
    reading a global is the point: the two must never be mixed up.

    Peak extra memory is set by row_chunk, not by the batch or the size of the user base.
    """
    ci, n = F["ci"], F["n"]
    ui, u_ok = lookup(uid, F["users"])
    idx = np.flatnonzero(F["c_ok"] & u_ok)

    uniq, inv = np.unique(ui[idx], return_inverse=True)
    Hb = H[uniq]
    deg = np.maximum(np.asarray(Hb.sum(axis=1)).ravel(), 1.0).astype(np.float32)
    Ub = Hb @ VEC                              # sum-pool, then unit length => dot = cosine
    Ub /= np.maximum(np.linalg.norm(Ub, axis=1, keepdims=True), 1e-12)
    Qb = (Hb @ TF).tocsr()                     # BM25 query = summed term counts
    Acat = np.asarray((Hb @ C_CAT).todense(), dtype=np.float32) / deg[:, None]
    Asub = np.asarray((Hb @ C_SUB).todense(), dtype=np.float32) / deg[:, None]

    sem = np.zeros(n, np.float32); bm = np.zeros(n, np.float32)
    cat = np.zeros(n, np.float32); sub = np.zeros(n, np.float32)
    for st in range(0, len(idx), row_chunk):
        sl = idx[st:st + row_chunk]
        c, v = ci[sl], inv[st:st + row_chunk]
        sem[sl] = np.einsum("ij,ij->i", VEC[c], Ub[v])
        bm[sl] = np.asarray(BM25_W[c].multiply(Qb[v]).sum(axis=1)).ravel()
        cat[sl] = Acat[v, CAT_CODE[c]]
        sub[sl] = Asub[v, SUB_CODE[c]]
    del Hb, Ub, Qb, Acat, Asub, ui, u_ok, idx, uniq, inv

    return {"pv": zscore(F["pv"]), "iv": zscore(F["iv"]), "ctr": zscore(F["ctr"]),
            "age": zscore(F["age_score"]), "sem": zscore(sem), "bm25": zscore(bm),
            "cat": zscore(cat), "sub": zscore(sub), "sent": zscore(F["sent"])}


def segment_ranks(scores, lens):
    """Ordinal descending rank within each impression, ties broken by slate position.

    Exploded rows are already contiguous per impression, so one stable lexsort does what
    a sort + rank().over() + group_by was doing, without leaving numpy.
    """
    seg = np.repeat(np.arange(len(lens)), lens)
    order = np.lexsort((-np.asarray(scores, dtype=np.float64), seg))
    ranks = np.empty(len(seg), dtype=np.int32)
    ranks[order] = (np.arange(len(seg)) - np.repeat(np.concatenate([[0], np.cumsum(lens)[:-1]]),
                                                    lens) + 1)
    return ranks

click matrices in 8s
   train (training lookback): 788,090 users, 18,388,808 clicks, 0.15 GB
   eval (validation lookback): 791,582 users, 18,371,402 clicks, 0.15 GB
   pred (test lookback)      : 807,677 users, 18,728,286 clicks, 0.15 GB


## 8. Two evaluation regimes, and the single-feature ablation

FRESH uses a CTR table that includes half of validation; STALE uses train only. Both are
scored on the second half of validation. The gap between them tells us how much each
signal decays — which on MIND turned out to be the difference between a good submission
and a mediocre one.

In [33]:
EVAL_N = 60_000
val_late = bh_val.filter(pl.col("impression_time") >= VAL_MID).head(EVAL_N)
print(f"evaluating on {val_late.height:,} impressions from the second half of validation")

t0 = time.time()
FRAMES, COMP = {}, {}
for regime, ctrv in [("FRESH", CTRV_FRESH), ("STALE", CTRV_STALE)]:
    FRAMES[regime] = prepare(val_late, ctrv, True)
    COMP[regime] = components(FRAMES[regime], UID_EVAL, H_EVAL)
    print(f"   {regime}: {FRAMES[regime]['n']:,} rows ({time.time()-t0:.0f}s)")
    mem(f"after {regime} components")


def evaluate(F, score):
    return score_fast(F["imp"], F["label"], score)


rows = []
for regime in ["FRESH", "STALE"]:
    f, c = FRAMES[regime], COMP[regime]
    singles = {"random": np.random.default_rng(0).random(f["n"]),
               "total_pageviews": c["pv"], "total_inviews": c["iv"],
               "decayed CTR": c["ctr"], "age bucket": c["age"],
               "semantic (contrastive)": c["sem"], "BM25": c["bm25"],
               "category affinity": c["cat"], "subcategory affinity": c["sub"],
               "sentiment": c["sent"]}
    for name, s in singles.items():
        m = evaluate(f, s); m["regime"] = regime; m["ranker"] = name
        rows.append(m)

board = (pl.DataFrame(rows).select(["regime", "ranker", "AUC", "MRR", "nDCG@5", "nDCG@10", "n"])
           .sort(["regime", "AUC"], descending=[False, True]))
print(board)

evaluating on 60,000 impressions from the second half of validation
   FRESH: 736,204 rows (5s)
   [mem] 17.58 GB |  20.9 GB free on /kaggle/working   after FRESH components
   STALE: 736,204 rows (10s)
   [mem] 17.66 GB |  20.9 GB free on /kaggle/working   after STALE components
shape: (20, 7)
┌────────┬────────────────────────┬──────────┬──────────┬──────────┬──────────┬───────┐
│ regime ┆ ranker                 ┆ AUC      ┆ MRR      ┆ nDCG@5   ┆ nDCG@10  ┆ n     │
│ ---    ┆ ---                    ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---   │
│ str    ┆ str                    ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ i64   │
╞════════╪════════════════════════╪══════════╪══════════╪══════════╪══════════╪═══════╡
│ FRESH  ┆ age bucket             ┆ 0.670567 ┆ 0.433936 ┆ 0.493644 ┆ 0.545081 ┆ 60000 │
│ FRESH  ┆ total_pageviews        ┆ 0.611923 ┆ 0.390998 ┆ 0.436676 ┆ 0.506391 ┆ 60000 │
│ FRESH  ┆ decayed CTR            ┆ 0.572934 ┆ 0.347618 ┆ 0.392148 ┆ 0.46799  ┆ 60000 │


## 9. Tune the blend on the STALE regime

Two searches. The first uses everything; the second excludes `total_pageviews`, because it
partly encodes the future (clicks in the 7 days after publication) and Q9 asks for metrics
with and without features unavailable at serving time. We submit the first and report both.

In [ ]:
from itertools import product

def blend(c, w):
    return sum(w[k] * c[k] for k in w)


def search(grid, comp, frame, label):
    keys = list(grid)
    best = None
    for vals in product(*(grid[k] for k in keys)):
        w = dict(zip(keys, vals))
        if all(v == 0 for v in w.values()):
            continue
        auc = evaluate(frame, blend(comp, w))["AUC"]
        if best is None or auc > best[0]:
            best = (auc, w)
    print(f"{label}: AUC {best[0]:.4f}  {best[1]}")
    return best


FULL = {"pv": [0.0, 1.0, 2.0, 3.0], "ctr": [0.0, 0.5, 1.0],
        "age": [0.0, 1.0, 2.0, 3.0], "sem": [0.0, 1.0, 2.0, 3.0],
        "bm25": [0.0, 0.5], "sub": [0.0, 0.5, 1.0]}
t0 = time.time()
BEST_AUC, BEST_W = search(FULL, COMP["STALE"], FRAMES["STALE"], "with pageviews   ")

HONEST = dict(FULL); HONEST["pv"] = [0.0]
HON_AUC, HON_W = search(HONEST, COMP["STALE"], FRAMES["STALE"], "without pageviews")
print(f"\nsearched in {time.time()-t0:.0f}s")
print(f"leakage-style gain from total_pageviews: {BEST_AUC - HON_AUC:+.4f} AUC")
print(f"\nsame weights on FRESH: {evaluate(FRAMES['FRESH'], blend(COMP['FRESH'], BEST_W))}")

with pageviews   : AUC 0.6806  {'pv': 3.0, 'ctr': 0.5, 'age': 3.0, 'sem': 1.0, 'bm25': 0.5, 'sub': 0.5}
without pageviews: AUC 0.6733  {'pv': 0.0, 'ctr': 1.0, 'age': 3.0, 'sem': 1.0, 'bm25': 0.5, 'sub': 0.5}

searched in 295s
leakage-style gain from total_pageviews: +0.0073 AUC

same weights on FRESH: {'AUC': np.float64(0.6833888539995563), 'MRR': np.float64(0.4545805072228027), 'nDCG@5': np.float64(0.5096101272345122), 'nDCG@10': np.float64(0.5609060093368072), 'n': 60000}


# A2 — EB-NeRD: CatBoost Learning-to-Rank Re-ranker

This section replaces the unsuccessful pure NRMS experiment with a **GBDT learning-to-rank re-ranker**. The A1 hybrid remains the candidate-generation/reference system. CatBoost is trained on impression groups using the same-impression clicked article as the positive label.

Features combine: A1 ranking signals, pretrained contrastive semantic/history features, article freshness/popularity, category affinity, session depth and candidate position. The final test pass scores the A1 candidate slate and outputs the required permutation.

**Important Q9 note:** `total_pageviews` is retained in the main leaderboard model because it was part of the A1 leaderboard pipeline, but it is explicitly reported as a serving-unavailable feature because the dataset definition makes it partly future-looking. A feature-masked evaluation is included.


In [ ]:
# ============================================================
# 11.1 CatBoost configuration
# ============================================================
SEED = 2026
np.random.seed(SEED)

from catboost import CatBoostRanker, Pool

CAT_TRAIN_IMPRESSIONS = min(300_000, bh_train.height)
CAT_EVAL_IMPRESSIONS = 60_000
CAT_ITERATIONS = 700
CAT_DEPTH = 8
CAT_LR = 0.08
CAT_L2 = 5.0
CAT_RANDOM_STRENGTH = 1.0
CAT_THREAD_COUNT = max(1, os.cpu_count() or 2)
CAT_RANK_LOSS = "PairLogit"

CAT_VAL_DF = val_late.head(CAT_EVAL_IMPRESSIONS)
CAT_MODEL_PATH = WORK / "ebnerd_a2_catboost_ranker.cbm"
CAT_OUT_TXT = SCRATCH / "predictions_ebnerd_a2_catboost.txt"
CAT_ZIP = WORK / "ebnerd_a2_catboost_predictions.zip"

FEATURE_NAMES = [
    "a1_score", "log_pageviews", "log_inviews", "ctr", "age_bucket",
    "semantic_history", "bm25_history", "category_affinity",
    "subcategory_affinity", "sentiment", "position_norm",
    "session_depth", "history_len"
]
print({"train_impressions":CAT_TRAIN_IMPRESSIONS,"eval_impressions":CAT_EVAL_IMPRESSIONS,
       "iterations":CAT_ITERATIONS,"depth":CAT_DEPTH,"learning_rate":CAT_LR,
       "loss":CAT_RANK_LOSS,"features":len(FEATURE_NAMES)})


{'train_impressions': 300000, 'eval_impressions': 60000, 'iterations': 700, 'depth': 8, 'learning_rate': 0.08, 'loss': 'PairLogit', 'features': 13}


In [ ]:
# ============================================================
# 11.2 Feature matrix for the GBDT ranker
# ============================================================
def _position_norm(lens):
    out=np.empty(int(np.sum(lens)),dtype=np.float32)
    p=0
    for k in lens.astype(np.int64):
        if k:
            out[p:p+k]=np.arange(k,dtype=np.float32)/max(k-1,1)
            p+=k
    return out

def _session_values(F, session_map):
    return np.fromiter((session_map.get(int(x),0) for x in F["imp"]),dtype=np.float32,count=F["n"])

def _history_len(F, uid, H):
    ui, ok=lookup(uid,F["users"])
    vals=np.zeros(F["n"],dtype=np.float32)
    good=ok
    if np.any(good):
        deg=np.asarray(H.sum(axis=1)).ravel().astype(np.float32)
        vals[good]=deg[ui[good]]
    return vals

def make_cat_features(F,C,uid,H,session_map):
    # Components are already z-scored by the A1 pipeline; the tree only needs their
    # relative ordering.  a1_score is the established A1 hybrid score.
    a1=blend(C,BEST_W).astype(np.float32)
    pos=_position_norm(F["lens"])
    sess=_session_values(F,session_map)
    hlen=_history_len(F,uid,H)
    X=np.column_stack([
        a1, C["pv"], C["iv"], C["ctr"], C["age"], C["sem"], C["bm25"],
        C["cat"], C["sub"], C["sent"], pos, sess, hlen
    ]).astype(np.float32,copy=False)
    assert X.shape[1]==len(FEATURE_NAMES)
    return X

# Build the labelled training frame and its user/history-derived features.
t0=time.time()
TRAIN_DF=bh_train.sample(n=CAT_TRAIN_IMPRESSIONS,seed=SEED,shuffle=True)
TRAIN_FRAME=prepare(TRAIN_DF,CTRV_STALE,True)
TRAIN_COMP=components(TRAIN_FRAME,UID_TRAIN,H_TRAIN)
X_train=make_cat_features(TRAIN_FRAME,TRAIN_COMP,UID_TRAIN,H_TRAIN,SESSION_DEPTH_TRAIN)
y_train=TRAIN_FRAME["label"].astype(np.float32)
g_train=TRAIN_FRAME["imp"].astype(np.int64)
print(f"training rows={len(y_train):,}; positives={int(y_train.sum()):,}; groups={TRAIN_DF.height:,}; time={time.time()-t0:.1f}s")
print("X_train:",X_train.shape,"memory MB:",X_train.nbytes/1e6)


training rows=3,324,782; positives=301,064; groups=300,000; time=26.6s
X_train: (3324782, 13) memory MB: 172.888664


In [ ]:
# ============================================================
# 11.3 Train CatBoost PairLogit ranker
# ============================================================
train_pool=Pool(X_train,label=y_train,group_id=g_train,feature_names=FEATURE_NAMES)

CAT_MODEL=CatBoostRanker(
    loss_function=CAT_RANK_LOSS,
    iterations=CAT_ITERATIONS,
    depth=CAT_DEPTH,
    learning_rate=CAT_LR,
    l2_leaf_reg=CAT_L2,
    random_strength=CAT_RANDOM_STRENGTH,
    random_seed=SEED,
    thread_count=CAT_THREAD_COUNT,
    verbose=100,
    allow_writing_files=False,
)
t0=time.time()
CAT_MODEL.fit(train_pool)
CAT_MODEL.save_model(str(CAT_MODEL_PATH))
print(f"CatBoost trained in {time.time()-t0:.1f}s")
print("saved:",CAT_MODEL_PATH)
print("feature importance:")
for n,v in sorted(zip(FEATURE_NAMES,CAT_MODEL.get_feature_importance()),key=lambda x:-x[1]):
    print(f"  {n:<24} {v:8.3f}")
# Release the large training matrix before validation/test work.
del X_train,y_train,g_train,train_pool,TRAIN_COMP,TRAIN_FRAME,TRAIN_DF
gc.collect()
mem("after CatBoost training")


0:	learn: 0.6617312	total: 2.59s	remaining: 30m 11s
100:	learn: 0.4487440	total: 4m 3s	remaining: 24m 5s
200:	learn: 0.4305973	total: 8m 4s	remaining: 20m 2s
300:	learn: 0.4177057	total: 12m 5s	remaining: 16m 1s
400:	learn: 0.4090547	total: 16m 4s	remaining: 11m 59s
500:	learn: 0.4023352	total: 20m 6s	remaining: 7m 59s
600:	learn: 0.3969143	total: 24m 7s	remaining: 3m 58s
699:	learn: 0.3921448	total: 28m 6s	remaining: 0us
CatBoost trained in 1688.5s
saved: /kaggle/working/ebnerd_a2_catboost_ranker.cbm
feature importance:


CatBoostError: Feature importance type EFstrType.LossFunctionChange requires training dataset                             to be passed to this function.

## 11.4 Validation: A1 hybrid vs CatBoost re-ranker


In [40]:
t0=time.time()
VAL_FRAME=FRAMES["STALE"]
VAL_COMP=COMP["STALE"]
X_val=make_cat_features(VAL_FRAME,VAL_COMP,UID_EVAL,H_EVAL,SESSION_DEPTH_EVAL)
CAT_SCORE=CAT_MODEL.predict(X_val).astype(np.float32)
A1_SCORE=blend(VAL_COMP,BEST_W).astype(np.float32)
A1_METRICS=score_fast(VAL_FRAME["imp"],VAL_FRAME["label"],A1_SCORE)
CAT_METRICS=score_fast(VAL_FRAME["imp"],VAL_FRAME["label"],CAT_SCORE)
print("A1 reference:",A1_METRICS)
print("CatBoost:",CAT_METRICS)
print("delta CatBoost-A1:",{k:CAT_METRICS[k]-A1_METRICS[k] for k in ["AUC","MRR","nDCG@5","nDCG@10"]})
print(f"validation scoring time={time.time()-t0:.1f}s")


A1 reference: {'AUC': np.float64(0.680599792284086), 'MRR': np.float64(0.45310796224003896), 'nDCG@5': np.float64(0.5080183353519172), 'nDCG@10': np.float64(0.5591101079662995), 'n': 60000}
CatBoost: {'AUC': np.float64(0.7128684056856752), 'MRR': np.float64(0.4778912484255546), 'nDCG@5': np.float64(0.5436129936313929), 'nDCG@10': np.float64(0.5854824720692581), 'n': 60000}
delta CatBoost-A1: {'AUC': np.float64(0.03226861340158926), 'MRR': np.float64(0.02478328618551562), 'nDCG@5': np.float64(0.03559465827947572), 'nDCG@10': np.float64(0.026372364102958623)}
validation scoring time=1.4s


## 11.5 Ablations and serving-safe view

Two checks are reported without retraining: (1) mask the history-derived features to measure the personalization/semantic contribution, and (2) mask `log_pageviews` to expose the serving-unavailable feature dependency. These are **inference masking ablations**, not independently retrained models.


In [41]:
# Feature-masking ablations on the same trained model.
def masked_metrics(mask_cols):
    Xm=X_val.copy()
    for name in mask_cols:
        Xm[:,FEATURE_NAMES.index(name)]=0.0
    s=CAT_MODEL.predict(Xm).astype(np.float32)
    return score_fast(VAL_FRAME["imp"],VAL_FRAME["label"],s),s

CAT_NO_HISTORY_METRICS,CAT_NO_HISTORY_SCORE=masked_metrics(["semantic_history","bm25_history","category_affinity","subcategory_affinity","history_len"])
CAT_NO_PV_METRICS,CAT_NO_PV_SCORE=masked_metrics(["log_pageviews"])
print("CatBoost full:",CAT_METRICS)
print("history-masked:",CAT_NO_HISTORY_METRICS)
print("pageviews-masked (serving-safe diagnostic):",CAT_NO_PV_METRICS)


CatBoost full: {'AUC': np.float64(0.7128684056856752), 'MRR': np.float64(0.4778912484255546), 'nDCG@5': np.float64(0.5436129936313929), 'nDCG@10': np.float64(0.5854824720692581), 'n': 60000}
history-masked: {'AUC': np.float64(0.705878932451462), 'MRR': np.float64(0.4670727260039452), 'nDCG@5': np.float64(0.5349886939849194), 'nDCG@10': np.float64(0.5770893769797354), 'n': 60000}
pageviews-masked (serving-safe diagnostic): {'AUC': np.float64(0.672759586160824), 'MRR': np.float64(0.43269542445388925), 'nDCG@5': np.float64(0.49079516093091413), 'nDCG@10': np.float64(0.5460865948110089), 'n': 60000}


In [43]:
# ============================================================
# Bootstrap + extended-evaluation helpers
# ============================================================
def bootstrap_pair(df,score_a,score_b,reps=1000,seed=SEED):
    # Impression-level paired bootstrap: both systems are evaluated on exactly the same
    # impression and candidate slate on every resample.
    lens=df["article_ids_inview"].list.len().to_numpy(); offsets=np.cumsum(np.r_[0,lens])
    per_a=[]; per_b=[]
    for i,r in enumerate(df.iter_rows(named=True)):
        sl=slice(offsets[i],offsets[i+1])
        clicked=set(r["article_ids_clicked"] or [])
        arts=r["article_ids_inview"] or []
        y=np.asarray([int(a in clicked) for a in arts],np.int8)
        if y.sum()==0 or y.sum()==len(y):
            continue
        per_a.append(score_fast(np.zeros(len(y),dtype=np.int64),y,score_a[sl]))
        per_b.append(score_fast(np.zeros(len(y),dtype=np.int64),y,score_b[sl]))
    keys=["AUC","MRR","nDCG@5","nDCG@10"]; rng=np.random.default_rng(seed); out=[]
    for k in keys:
        va=np.asarray([x[k] for x in per_a]); vb=np.asarray([x[k] for x in per_b]); delta=vb-va
        boot=np.empty(reps)
        for r in range(reps): boot[r]=delta[rng.integers(0,len(delta),len(delta))].mean()
        lo,hi=np.quantile(boot,[.025,.975])
        out.append({"metric":k,"delta_b_minus_a":float(delta.mean()),"ci_low":float(lo),"ci_high":float(hi),"excludes_zero":bool(lo>0 or hi<0)})
    return pl.DataFrame(out)

# Extended metrics required by the assignment.
train_click_counts=defaultdict(int)
for xs in bh_train["article_ids_clicked"].to_list():
    for a in (xs or []): train_click_counts[int(a)]+=1
click_arr=np.array([train_click_counts.get(int(a),0) for a in ART_IDS],np.int64)
q80=np.quantile(click_arr[click_arr>0],.80) if np.any(click_arr>0) else 0
q20=np.quantile(click_arr[click_arr>0],.20) if np.any(click_arr>0) else 0
CAT_BY_ART={int(a):int(CAT_CODE[j]) for j,a in enumerate(ART_IDS)}

def beyond_accuracy(df,score):
    rows=[]; seen=set(); total_clicks=max(sum(train_click_counts.values()),1)
    lens=df["article_ids_inview"].list.len().to_numpy(); offsets=np.cumsum(np.r_[0,lens])
    for i,r in enumerate(df.iter_rows(named=True)):
        arts=r["article_ids_inview"] or []; sc=score[offsets[i]:offsets[i+1]]; order=np.argsort(-sc,kind="stable")[:10]; top=[arts[j] for j in order]
        cats=[CAT_BY_ART.get(int(a),-1) for a in top]; pairs=max(len(cats)*(len(cats)-1)/2,1)
        same=sum(cats[x]==cats[y] for x in range(len(cats)) for y in range(x+1,len(cats)))
        div=1-same/pairs
        probs=[max(train_click_counts.get(int(a),0)/total_clicks,1e-12) for a in top]
        nov=float(np.mean([-np.log2(p) for p in probs])) if probs else 0
        seen.update(int(a) for a in top); rows.append((div,nov))
    return {"diversity@10":float(np.mean([x[0] for x in rows])),"novelty@10":float(np.mean([x[1] for x in rows])),"coverage@10":len(seen)/len(CATALOGUE)}

val_hlen={int(r["user_id"]):len(r["hist"] or []) for r in HIST_EVAL.iter_rows(named=True)}
def slice_metrics(df,score,mode):
    keep=[]
    for i,r in enumerate(df.iter_rows(named=True)):
        if mode=="cold": ok=val_hlen.get(int(r["user_id"]),0)<5
        elif mode=="warm": ok=val_hlen.get(int(r["user_id"]),0)>=5
        elif mode=="head": ok=any(train_click_counts.get(int(a),0)>=q80 for a in (r["article_ids_clicked"] or []))
        elif mode=="tail": ok=any(train_click_counts.get(int(a),0)<=q20 for a in (r["article_ids_clicked"] or []))
        else: ok=True
        if ok: keep.append(i)
    if not keep:return None,0
    lens=df["article_ids_inview"].list.len().to_numpy(); offsets=np.cumsum(np.r_[0,lens]); ids=[]; labs=[]; flat=[]
    for i in keep:
        r=df.row(i,named=True); arts=r["article_ids_inview"] or []; flat.extend(score[offsets[i]:offsets[i+1]]); ids.extend([r["impression_id"]]*len(arts)); labs.extend([int(a in set(r["article_ids_clicked"] or [])) for a in arts])
    return score_fast(np.asarray(ids),np.asarray(labs),np.asarray(flat)),len(keep)


## 11.6 Paired bootstrap 95% CIs


In [44]:
BOOT_CAT_VS_A1=bootstrap_pair(CAT_VAL_DF,A1_SCORE,CAT_SCORE,reps=1000,seed=SEED)
BOOT_HISTORY=bootstrap_pair(CAT_VAL_DF,CAT_NO_HISTORY_SCORE,CAT_SCORE,reps=1000,seed=SEED+1)
BOOT_PV=bootstrap_pair(CAT_VAL_DF,CAT_NO_PV_SCORE,CAT_SCORE,reps=1000,seed=SEED+2)
print("CatBoost - A1"); print(BOOT_CAT_VS_A1)
print("CatBoost full - history masked"); print(BOOT_HISTORY)
print("CatBoost full - pageviews masked"); print(BOOT_PV)


CatBoost - A1
shape: (4, 5)
┌─────────┬─────────────────┬──────────┬──────────┬───────────────┐
│ metric  ┆ delta_b_minus_a ┆ ci_low   ┆ ci_high  ┆ excludes_zero │
│ ---     ┆ ---             ┆ ---      ┆ ---      ┆ ---           │
│ str     ┆ f64             ┆ f64      ┆ f64      ┆ bool          │
╞═════════╪═════════════════╪══════════╪══════════╪═══════════════╡
│ AUC     ┆ 0.032269        ┆ 0.030164 ┆ 0.03435  ┆ true          │
│ MRR     ┆ 0.024783        ┆ 0.022108 ┆ 0.027611 ┆ true          │
│ nDCG@5  ┆ 0.035595        ┆ 0.032976 ┆ 0.038386 ┆ true          │
│ nDCG@10 ┆ 0.026372        ┆ 0.024105 ┆ 0.028673 ┆ true          │
└─────────┴─────────────────┴──────────┴──────────┴───────────────┘
CatBoost full - history masked
shape: (4, 5)
┌─────────┬─────────────────┬──────────┬──────────┬───────────────┐
│ metric  ┆ delta_b_minus_a ┆ ci_low   ┆ ci_high  ┆ excludes_zero │
│ ---     ┆ ---             ┆ ---      ┆ ---      ┆ ---           │
│ str     ┆ f64             ┆ f64      ┆ f6

## 11.7 Extended evaluation


In [45]:
# Beyond-accuracy and head/tail/cold/warm slices for the full CatBoost model.
BA_METRICS=beyond_accuracy(CAT_VAL_DF,CAT_SCORE)
print("beyond accuracy",BA_METRICS)
for mode in ["cold","warm","head","tail"]:
    m,n=slice_metrics(CAT_VAL_DF,CAT_SCORE,mode)
    print(mode,n,m)


beyond accuracy {'diversity@10': 0.776102552910053, 'novelty@10': 37.07294657849009, 'coverage@10': 0.019427915979640116}
cold 0 None
warm 60000 {'AUC': np.float64(0.7128684056856752), 'MRR': np.float64(0.4778912484255545), 'nDCG@5': np.float64(0.5436129936313929), 'nDCG@10': np.float64(0.5854824720692581), 'n': 60000}
head 835 {'AUC': np.float64(0.38628681736523895), 'MRR': np.float64(0.25016843440683384), 'nDCG@5': np.float64(0.23937585546325152), 'nDCG@10': np.float64(0.3132174138074969), 'n': 835}
tail 58596 {'AUC': np.float64(0.7215444337588681), 'MRR': np.float64(0.4834700922281513), 'nDCG@5': np.float64(0.5506269315770073), 'nDCG@10': np.float64(0.5914673087189097), 'n': 58596}


## 11.8 Serving benchmark and 10× scale


In [46]:
# Benchmark actual tree scoring for one impression at a time. Candidate generation/feature
# construction is inherited from A1; this measures the CatBoost rerank stage itself.
def cat_score_one(row):
    f=prepare(row,CTRV_STALE,False)
    c=components(f,UID_EVAL,H_EVAL)
    x=make_cat_features(f,c,UID_EVAL,H_EVAL,SESSION_DEPTH_EVAL)
    return CAT_MODEL.predict(x).astype(np.float32)

def bench_cat(n=100):
    rng=np.random.default_rng(SEED); vals=[]
    for _ in range(n):
        row=CAT_VAL_DF.slice(int(rng.integers(CAT_VAL_DF.height)),1)
        t=time.perf_counter(); cat_score_one(row); vals.append((time.perf_counter()-t)*1000)
    return {"p50_ms":float(np.percentile(vals,50)),"p95_ms":float(np.percentile(vals,95)),"p99_ms":float(np.percentile(vals,99))}
SERV_CAT=bench_cat(); print("CatBoost rerank latency",SERV_CAT)
print("CatBoost model size MB:",CAT_MODEL_PATH.stat().st_size/1e6)
print("Frozen article vectors MB:",VEC.nbytes/1e6)
print("10x catalogue vector-store GB:",10*VEC.nbytes/1e9)
print("10x traffic note: tree scoring is O(candidate_rows * trees * depth); candidate/feature generation and memory bandwidth become the dominant scaling pressures.")


CatBoost rerank latency {'p50_ms': 33.22834399978092, 'p95_ms': 35.44387829974767, 'p99_ms': 38.40911323030015}
CatBoost model size MB: 2.953368
Frozen article vectors MB: 385.661952
10x catalogue vector-store GB: 3.85661952
10x traffic note: tree scoring is O(candidate_rows * trees * depth); candidate/feature generation and memory bandwidth become the dominant scaling pressures.


## 11.9 Final EB-NeRD test prediction — CatBoost re-ranker


In [47]:
# Score the full test candidate slate in bounded batches. Test uses the test-specific history
# and the final CTR table built only from labels before the test week.
PRED_CANDIDATE_ROWS=300_000
_lens=bh_test["article_ids_inview"].list.len().to_numpy().astype(np.int64)
grp=(np.cumsum(_lens)-1)//PRED_CANDIDATE_ROWS
starts=np.flatnonzero(np.diff(grp,prepend=np.int64(-1)))
sizes=np.diff(np.append(starts,len(_lens)))

def cat_test_scores(batch):
    f=prepare(batch,CTRV_FINAL,False)
    c=components(f,UID_PRED,H_PRED)
    x=make_cat_features(f,c,UID_PRED,H_PRED,SESSION_DEPTH_PRED)
    s=CAT_MODEL.predict(x).astype(np.float32)
    return f["imp"],f["lens"],s

if CAT_OUT_TXT.exists(): CAT_OUT_TXT.unlink()
written=0; t0=time.time()
with open(CAT_OUT_TXT,"w",encoding="utf-8",newline="\n") as fh:
    for bi,(st,sz) in enumerate(zip(starts,sizes)):
        batch=bh_test.slice(int(st),int(sz))
        imp,lens,s=cat_test_scores(batch)
        ranks=segment_ranks(s,lens)
        out=[]; p=0
        for iid,k in zip(imp.tolist(),lens.tolist()):
            out.append(f"{iid} [{','.join(map(str,ranks[p:p+k].tolist()))}]")
            p+=k
        fh.write("\n".join(out)+"\n")
        written+=len(out)
        if bi%25==0: print(f"{written:,}/{bh_test.height:,} ({100*written/bh_test.height:.1f}%) elapsed {time.time()-t0:.0f}s")
print("written",written,"lines",CAT_OUT_TXT.stat().st_size/1e6,"MB")


25,603/13,536,710 (0.2%) elapsed 2s
667,728/13,536,710 (4.9%) elapsed 60s
1,309,369/13,536,710 (9.7%) elapsed 117s
1,950,118/13,536,710 (14.4%) elapsed 175s
2,591,203/13,536,710 (19.1%) elapsed 232s
3,230,359/13,536,710 (23.9%) elapsed 290s
3,876,065/13,536,710 (28.6%) elapsed 347s
4,519,755/13,536,710 (33.4%) elapsed 405s
5,162,347/13,536,710 (38.1%) elapsed 462s
5,808,084/13,536,710 (42.9%) elapsed 519s
6,448,312/13,536,710 (47.6%) elapsed 575s
7,090,028/13,536,710 (52.4%) elapsed 632s
7,729,887/13,536,710 (57.1%) elapsed 688s
8,365,243/13,536,710 (61.8%) elapsed 746s
9,007,015/13,536,710 (66.5%) elapsed 805s
9,648,563/13,536,710 (71.3%) elapsed 864s
10,294,810/13,536,710 (76.1%) elapsed 923s
10,936,817/13,536,710 (80.8%) elapsed 982s
11,575,924/13,536,710 (85.5%) elapsed 1041s
12,214,570/13,536,710 (90.2%) elapsed 1100s
12,851,575/13,536,710 (94.9%) elapsed 1159s
13,344,206/13,536,710 (98.6%) elapsed 1216s
13,404,206/13,536,710 (99.0%) elapsed 1316s
13,434,206/13,536,710 (99.2%) ela

In [50]:
# ============================================================
# REPAIR EXISTING CATBOOST TEST OUTPUT
# DO NOT RESCORE THE MODEL
# ============================================================

import json
import time
from pathlib import Path

OLD_OUT = CAT_OUT_TXT
FIXED_OUT = Path("/kaggle/working/ebnerd_a2_catboost_predictions_fixed.txt")

expected_ids = bh_test["impression_id"].to_list()
expected_len = (
    bh_test["article_ids_inview"]
    .list.len()
    .to_list()
)

t0 = time.time()

line_count = 0
problems = []

with open(OLD_OUT, "r", encoding="utf-8") as fin, \
     open(FIXED_OUT, "w", encoding="utf-8") as fout:

    for i, line in enumerate(fin):

        line = line.strip()

        if not line:
            problems.append(f"line {i+1}: empty")
            continue

        try:
            old_iid, ranks_text = line.split(maxsplit=1)
            ranks = json.loads(ranks_text)
        except Exception as e:
            problems.append(f"line {i+1}: {e}")
            continue

        # Keep the already-computed ranks.
        # Replace ONLY the incorrect impression ID.
        if len(ranks) != expected_len[i]:
            problems.append(
                f"line {i+1}: rank length {len(ranks)} "
                f"!= expected {expected_len[i]}"
            )
            if len(problems) > 10:
                break
            continue

        if sorted(ranks) != list(range(1, len(ranks) + 1)):
            problems.append(
                f"line {i+1}: ranks are not a permutation"
            )
            if len(problems) > 10:
                break
            continue

        fout.write(
            f"{expected_ids[i]} {ranks_text}\n"
        )

        line_count += 1

if problems:
    print("Problems found:")
    for p in problems[:10]:
        print(p)
    raise RuntimeError("Repair stopped because validation failed.")

assert line_count == len(expected_ids)

print(
    f"Repaired {line_count:,} lines "
    f"in {time.time()-t0:.1f}s"
)
print("Fixed file:", FIXED_OUT)
print(
    "Size:",
    FIXED_OUT.stat().st_size / 1e6,
    "MB"
)

Repaired 13,536,710 lines in 73.9s
Fixed file: /kaggle/working/ebnerd_a2_catboost_predictions_fixed.txt
Size: 703.062222 MB


In [51]:
# ============================================================
# VALIDATE REPAIRED OUTPUT
# ============================================================

problems = []

with open(FIXED_OUT, encoding="utf-8") as fh:

    for i, line in enumerate(fh):

        try:
            iid, ranks_text = line.strip().split(maxsplit=1)
            ranks = json.loads(ranks_text)
        except Exception as e:
            problems.append(f"line {i+1}: {e}")
            continue

        if iid != str(expected_ids[i]):
            problems.append(
                f"line {i+1}: wrong ID "
                f"(got {iid}, expected {expected_ids[i]})"
            )

        if len(ranks) != expected_len[i]:
            problems.append(
                f"line {i+1}: wrong length"
            )

        if sorted(ranks) != list(range(1, len(ranks)+1)):
            problems.append(
                f"line {i+1}: not permutation"
            )

        if len(problems) > 10:
            break

assert not problems, problems[:10]

with open(FIXED_OUT, encoding="utf-8") as fh:
    actual_lines = sum(1 for _ in fh)

assert actual_lines == len(expected_ids)

print("✅ PASSED repaired submission validation")
print("Lines:", actual_lines)

✅ PASSED repaired submission validation
Lines: 13536710


In [52]:
# ============================================================
# CREATE FINAL CODABENCH ZIP
# ============================================================

FINAL_ZIP = Path(
    "/kaggle/working/ebnerd_a2_catboost_predictions.zip"
)

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

with zipfile.ZipFile(
    FINAL_ZIP,
    "w",
    zipfile.ZIP_DEFLATED,
    compresslevel=6
) as z:

    z.write(
        FIXED_OUT,
        arcname="predictions.txt"
    )

with zipfile.ZipFile(FINAL_ZIP) as z:
    assert z.namelist() == ["predictions.txt"]

print("========================================")
print("✅ FINAL ZIP READY")
print("========================================")
print(FINAL_ZIP)
print(f"Size: {FINAL_ZIP.stat().st_size/1e6:.2f} MB")

✅ FINAL ZIP READY
/kaggle/working/ebnerd_a2_catboost_predictions.zip
Size: 229.97 MB


## 11.10 Report artifacts / A2 checklist


In [53]:
REPORT_DIR=WORK/"A2_CatBoost_report_artifacts"
REPORT_DIR.mkdir(parents=True,exist_ok=True)
comparison=pl.DataFrame([
    {"model":"A1_reference","AUC":A1_METRICS["AUC"],"MRR":A1_METRICS["MRR"],"nDCG@5":A1_METRICS["nDCG@5"],"nDCG@10":A1_METRICS["nDCG@10"]},
    {"model":"CatBoost_re_ranker","AUC":CAT_METRICS["AUC"],"MRR":CAT_METRICS["MRR"],"nDCG@5":CAT_METRICS["nDCG@5"],"nDCG@10":CAT_METRICS["nDCG@10"]},
    {"model":"CatBoost_history_masked","AUC":CAT_NO_HISTORY_METRICS["AUC"],"MRR":CAT_NO_HISTORY_METRICS["MRR"],"nDCG@5":CAT_NO_HISTORY_METRICS["nDCG@5"],"nDCG@10":CAT_NO_HISTORY_METRICS["nDCG@10"]},
    {"model":"CatBoost_pageviews_masked","AUC":CAT_NO_PV_METRICS["AUC"],"MRR":CAT_NO_PV_METRICS["MRR"],"nDCG@5":CAT_NO_PV_METRICS["nDCG@5"],"nDCG@10":CAT_NO_PV_METRICS["nDCG@10"]},
])
comparison.write_csv(REPORT_DIR/"model_comparison.csv")
BOOT_CAT_VS_A1.write_csv(REPORT_DIR/"bootstrap_catboost_vs_a1.csv")
BOOT_HISTORY.write_csv(REPORT_DIR/"bootstrap_history_ablation.csv")
BOOT_PV.write_csv(REPORT_DIR/"bootstrap_pageviews_ablation.csv")
summary={"dataset":"EB-NeRD large","model":"CatBoost PairLogit learning-to-rank re-ranker","train_impressions":CAT_TRAIN_IMPRESSIONS,"iterations":CAT_ITERATIONS,"depth":CAT_DEPTH,"learning_rate":CAT_LR,"loss":CAT_RANK_LOSS,"features":FEATURE_NAMES,"metrics":CAT_METRICS,"a1_reference":A1_METRICS,"history_masked":CAT_NO_HISTORY_METRICS,"pageviews_masked":CAT_NO_PV_METRICS,"serving":SERV_CAT,"submission":str(CAT_ZIP)}
with open(REPORT_DIR/"summary.json","w") as f: json.dump(summary,f,indent=2,default=lambda x:float(x) if isinstance(x,np.floating) else x)
print("saved artifacts:"); [print(p.name) for p in sorted(REPORT_DIR.iterdir())]


saved artifacts:
bootstrap_catboost_vs_a1.csv
bootstrap_history_ablation.csv
bootstrap_pageviews_ablation.csv
model_comparison.csv
summary.json


[None, None, None, None, None]